# Chapter 14 &mdash; Procedure vs. Algorithm, and the First Impossibility Result

**Concept 2 of the Chapter 14 decomposition:** *Procedure vs. Algorithm, and the First Impossibility Result*

An algorithm is a procedure that halts on <i>all</i> inputs &mdash; and no machine can tell which procedures are algorithms.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14-Interp/Concept-Procedure-Vs-Algorithm/Concept-Procedure-Vs-Algorithm.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Two words that are not synonyms:

* a **procedure** is any TM &mdash; it may halt or loop;
* an **algorithm** is a TM that **halts on every input**.

Every algorithm is a procedure; the converse fails. And here is the first impossibility
result, stated before it is proved:

> **No procedure decides whether a given procedure is an algorithm.**

This is the halting problem wearing different clothes, and it is why "is this function
total?" is not a question your compiler can answer. The rest of the chapter builds the
vocabulary to prove it.

## 2. Definitions

### Three machines: two algorithms and a procedure

In [ ]:
Alg1 = md2mc('''TM
I : 0 ; 0 , R -> F
I : 1 ; 1 , R -> D
I : . ; . , R -> D
''')
Alg2 = md2mc('''TM
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')
Proc = md2mc('''TM
I : 0 ; 0 , R -> F
I : 1 ; 1 , R -> L
I : . ; . , R -> L
L : 0 ; 0 , R -> L
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')

# --- thin wrappers over Jove's TM runner --------------------------------
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

### Testing totality -- and why the test can only ever be partial

In [ ]:
from itertools import product
def halts_on_all(T, upto=5, fuel=200, sigma='01'):
    for k in range(1, upto + 1):
        for p in product(sigma, repeat=k):
            if not tm_halts(T, ''.join(p), fuel=fuel):
                return False, ''.join(p)
    return True, None

<!-- nav-strip -->

---

&larr;&nbsp;[Ch14&nbsp;1.&nbsp;Why Study Impossibility Results?](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14-Interp/Concept-Why-Impossibility-Results/Concept-Why-Impossibility-Results.ipynb) &nbsp;&middot;&nbsp; [**Chapter 14** index](https://github.com/ganeshutah/Jove/blob/master/Chapter14-Interp/README.md) &nbsp;&middot;&nbsp; [Ch14&nbsp;3.&nbsp;The Language Families of Turing Machines: RE and Recursive](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14-Interp/Concept-RE-And-Recursive-Families/Concept-RE-And-Recursive-Families.ipynb)&nbsp;&rarr;

---

## 3. Tests

Two of the three halt on everything we try.

In [ ]:
for name, T in [('Alg1', Alg1), ('Alg2', Alg2), ('Proc', Proc)]:
    ok, wit = halts_on_all(T)
    print("  %-6s halts on all tested inputs? %-6s %s"
          % (name, ok, ('witness ' + repr(wit)) if wit else ''))
assert halts_on_all(Alg1)[0] and halts_on_all(Alg2)[0]
assert not halts_on_all(Proc)[0]

But **the test is a search, not a decision.** It can only ever find a witness.

In [ ]:
print("halts_on_all returns False only when it FINDS a diverging input.")
print("It returns True only because it gave up after a finite sweep.")
print()
print("Enlarging the sweep never turns it into a decision procedure:")
for upto in [3, 5, 7]:
    ok, _ = halts_on_all(Alg1, upto=upto)
    print("   tested up to length %d : %s -- still not a proof" % (upto, ok))

The vocabulary, precisely.

In [ ]:
print("procedure : any TM.  May halt, may loop.")
print("algorithm : a TM that halts on EVERY input.")
print()
print("recognizer (semi-decider) : implements a procedure")
print("decider                   : implements an algorithm")

And the result the chapter is heading for.

In [ ]:
print("CLAIM  no procedure decides, of an arbitrary procedure, whether it")
print("       is an algorithm.")
print()
print("If one existed you could ask it about a machine that loops exactly")
print("when some other machine fails to halt -- and you would have solved")
print("the halting problem.  Chapter 15 makes the reduction precise.")

## 4. Exercises


1. Is every DFA an algorithm? Every PDA? Every TM?
2. Give a procedure that is an algorithm on one alphabet but not on another.
3. Why does "test it on a million inputs" not settle totality?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 253 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter14-Interp/Concept-Procedure-Vs-Algorithm')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')